In [11]:
import pandas as pd
df = pd.read_parquet("../data/interim/facts_long")
print(df.shape)

(3024124, 9)


In [12]:
ecart = (df["filed"] - df["end"]).dt.days
d = df[ecart.between(0, 550)].copy()

print(f"Lignes gardées : {len(d):,} ({len(d)/len(df):.1%})")

Lignes gardées : 2,656,629 (87.8%)


In [13]:
d = d.sort_values(["cik", "concept", "end", "filed", "accn"])
d = d.drop_duplicates(subset=["cik", "concept", "end"], keep="first")

# Vérification : chaque (cik, concept, end) n'apparaît plus qu'une fois
print(f"Lignes : {len(d):,}")
print("Doublons restants :", d.duplicated(subset=["cik", "concept", "end"]).sum())

Lignes : 1,524,552
Doublons restants : 0


In [14]:
non_negatifs = [
    "Assets", "AssetsCurrent", "AssetsNoncurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
    "AccountsReceivableNetCurrent", "InventoryNet", "PrepaidExpenseCurrent",
    "PropertyPlantAndEquipmentNet", "Goodwill", "IntangibleAssetsNetExcludingGoodwill",
    "Liabilities", "LiabilitiesCurrent", "LiabilitiesNoncurrent",
    "AccountsPayableCurrent", "AccruedLiabilitiesCurrent",
    "DebtCurrent", "LongTermDebt", "LongTermDebtCurrent",
    "CommonStockValue", "LiabilitiesAndStockholdersEquity",
]
impossible = d["concept"].isin(non_negatifs) & (d["val"] < 0)
print("Valeurs négatives impossibles :", impossible.sum())

d.loc[impossible, "val"] = float("nan")

Valeurs négatives impossibles : 577


In [15]:
wide = d.pivot(index=["cik", "end"], columns="concept", values="val")
wide.columns.name = None
wide = wide.reset_index()

print(wide.shape)
wide[wide["cik"] == "0000001750"].head()

(116001, 27)


,cik,end,AccountsPayableCurrent,AccountsReceivableNetCurrent,AccruedLiabilitiesCurrent,AccumulatedOtherComprehensiveIncomeLossNetOfTax,AdditionalPaidInCapital,Assets,AssetsCurrent,AssetsNoncurrent,...,Liabilities,LiabilitiesAndStockholdersEquity,LiabilitiesCurrent,LiabilitiesNoncurrent,LongTermDebt,LongTermDebtCurrent,PrepaidExpenseCurrent,PropertyPlantAndEquipmentNet,RetainedEarningsAccumulatedDeficit,StockholdersEquity
5,0000001750,2010-05-31,114906000.0,238466000.0,108950000.0,-29646000.0,NaN,1.500181e+09,8.463310e+08,434204000.0,...,NaN,1.500181e+09,324689000.0,429142000.0,NaN,53292000.0,NaN,219646000.0,419287000.0,746906000.0
6,0000001750,2011-05-31,185096000.0,287435000.0,116839000.0,-18645000.0,NaN,1.703727e+09,9.139850e+08,465365000.0,...,NaN,1.703727e+09,416010000.0,452428000.0,NaN,11323000.0,NaN,324377000.0,486130000.0,835845000.0
7,0000001750,2012-05-31,201405000.0,302056000.0,148956000.0,-55190000.0,NaN,2.195653e+09,1.063272e+09,749448000.0,...,NaN,2.195653e+09,473226000.0,856405000.0,NaN,122200000.0,NaN,382933000.0,541772000.0,864649000.0
8,0000001750,2013-05-31,149300000.0,297400000.0,153300000.0,-42500000.0,NaN,2.136900e+09,1.033700e+09,741500000.0,...,NaN,2.136900e+09,389000000.0,828400000.0,NaN,86400000.0,NaN,361700000.0,584900000.0,918600000.0
9,0000001750,2014-05-31,171100000.0,283100000.0,161300000.0,-29300000.0,NaN,2.199500e+09,1.116900e+09,767700000.0,...,NaN,2.199500e+09,402100000.0,796700000.0,NaN,69700000.0,NaN,314900000.0,646000000.0,999500000.0


In [16]:
avant = len(wide)
wide = wide[wide["Assets"].notna()].copy()

print(f"Bilans : {avant:,} → {len(wide):,}")
print("Entreprises :", wide["cik"].nunique())

Bilans : 116,001 → 109,348
Entreprises : 13962


In [17]:
# Étape 5 bis — ne garder que les clôtures annuelles
# On repart des lignes Assets AVANT dédoublonnage : il faut voir toutes les dates de chaque rapport
a = df[ecart.between(0, 550) & (df["concept"] == "Assets")].copy()

# Date du bilan principal de chaque rapport = la date la plus récente qu'il contient
a["fin_rapport"] = a.groupby("accn")["end"].transform("max")
a["ecart_fin"] = (a["fin_rapport"] - a["end"]).dt.days

# Clôture de l'année (écart 0) ou comparatif N-1 (environ 1 an avant)
annuel = a[(a["ecart_fin"] == 0) | a["ecart_fin"].between(330, 400)]
dates_annuelles = annuel[["cik", "end"]].drop_duplicates()

avant = len(wide)
wide = wide.merge(dates_annuelles, on=["cik", "end"], how="inner")
print(f"Bilans : {avant:,} → {len(wide):,}  ({avant - len(wide):,} dates non annuelles supprimées)")

Bilans : 109,348 → 107,562  (1,786 dates non annuelles supprimées)


In [18]:
manquant = wide["Liabilities"].isna()

total = wide["LiabilitiesAndStockholdersEquity"].fillna(wide["Assets"])
wide.loc[manquant, "Liabilities"] = total[manquant] - wide.loc[manquant, "StockholdersEquity"]

# Une dette négative issue du calcul est incohérente -> NaN
wide.loc[wide["Liabilities"] < 0, "Liabilities"] = float("nan")

wide["liabilities_imputed"] = manquant & wide["Liabilities"].notna()
print(f"Dettes reconstruites : {wide['liabilities_imputed'].mean():.1%}")
print(f"Dettes encore manquantes : {wide['Liabilities'].isna().mean():.1%}")

Dettes reconstruites : 16.2%
Dettes encore manquantes : 2.0%


In [19]:
wide = wide.sort_values(["cik", "end"]).reset_index(drop=True)
wide["days_since_prev"] = wide.groupby("cik")["end"].diff().dt.days

ecarts = wide["days_since_prev"].dropna()
print(f"Écarts entre 330 et 400 j : {ecarts.between(330, 400).mean():.1%}")
print(ecarts.describe())

Écarts entre 330 et 400 j : 98.3%
count    93600.000000
mean       368.053622
std         95.935774
min          1.000000
25%        365.000000
50%        365.000000
75%        366.000000
max       4748.000000
Name: days_since_prev, dtype: float64


In [20]:
n = wide.groupby("cik").size().sort_values(ascending=False)
cik_max = n.index[0]
print(cik_max, n.iloc[0])
wide.loc[wide["cik"] == cik_max, ["end", "Assets", "days_since_prev"]].head(15)

0000833444 20


,end,Assets,days_since_prev
21506,2008-09-26,2.880400e+10,NaN
21507,2009-09-25,2.555300e+10,364.0
21508,2010-09-24,2.712800e+10,364.0
21509,2011-09-30,2.677700e+10,371.0
21510,2012-09-28,1.236500e+10,364.0
21511,2013-09-27,1.217600e+10,364.0
21512,2014-09-26,1.180900e+10,364.0
21513,2015-09-25,1.232100e+10,364.0
21514,2015-09-30,2.962200e+10,5.0
21515,2016-09-30,6.325300e+10,366.0


In [21]:
wide.to_parquet("../data/interim/balance_sheet_wide.parquet", index=False)